# Exploratory Data Analysis - Bureau Dataset

This notebook performs comprehensive exploratory data analysis (EDA) on the `bureau.csv` and `bureau_balance.csv` datasets from the Home Credit Default Risk competition.

**Scope:** Bureau data only (supplementary credit bureau data for applicants)

**Related Notebooks:**
- `application_eda.ipynb` - Analysis of Application_train dataset
- Additional datasets have their own dedicated notebooks

**Analysis Sections:**
1. Setup and Data Loading
2. Bureau Dataset Exploration
3. Bureau Balance Dataset Exploration
4. Merged Bureau Analysis
5. Missing Data Analysis
6. Numeric Features Analysis
7. Categorical Features Analysis
8. Data Quality Checks
9. Feature Engineering Opportunities

## 1. Setup and Data Loading

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

# Add src to path to import custom modules
sys.path.insert(0, "/Users/bharathregula/code/home_credit_risk_deeplearn/src")
from home_credit_risk.eda_utils import display_basic_info

# Configure polars display options
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)  # show all columns
pl.Config.set_fmt_str_lengths(100)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Load datasets (lazily — nothing is read from disk until .collect())
datasets = Path("/Users/bharathregula/code/datasets/home-credit-default-risk")

# scan_csv builds a LazyFrame; dtype overrides shrink memory up front.
# SK_ID_BUREAU is the join key and must share a dtype across both frames.
bureau_lf = pl.scan_csv(
    datasets / "bureau.csv",
    schema_overrides={"SK_ID_CURR": pl.Int32, "SK_ID_BUREAU": pl.Int32},
)
bureau_balance_lf = pl.scan_csv(
    datasets / "bureau_balance.csv",
    schema_overrides={
        "SK_ID_BUREAU": pl.Int32,
        "MONTHS_BALANCE": pl.Int8,
        "STATUS": pl.Categorical,
    },
)

# Row counts without materializing all columns
bureau_rows = bureau_lf.select(pl.len()).collect().item()
balance_rows = bureau_balance_lf.select(pl.len()).collect().item()

print("Datasets scanned successfully!")
print(f"Bureau rows: {bureau_rows}")
print(f"Bureau Balance rows: {balance_rows}")

In [ ]:
# Quick preview of bureau data (only 5 rows are materialized)
bureau_lf.head().collect()

In [ ]:
# Quick preview of bureau balance data
bureau_balance_lf.head().collect()

## 2. Bureau Dataset Exploration

In [ ]:
# Display basic information about bureau dataset
bureau_df = bureau_lf.collect()
display_basic_info(bureau_df, df_name="Bureau")

In [ ]:
# Summary statistics
bureau_df.describe()

## 3. Bureau Balance Dataset Exploration

In [ ]:
# Display basic information about bureau balance dataset
bureau_balance_df = bureau_balance_lf.collect()
display_basic_info(bureau_balance_df, df_name="Bureau_Balance")

In [ ]:
# Summary statistics
bureau_balance_df.describe()

In [ ]:
bureau_balance_lf.head().collect()

## 4. Merged Bureau Analysis

Merge bureau and bureau_balance data for combined analysis.

In [ ]:
# Merge bureau and bureau_balance on SK_ID_BUREAU (lazy left join)
bureau_merged_lf = bureau_lf.join(bureau_balance_lf, on="SK_ID_BUREAU", how="left")

merged_rows = bureau_merged_lf.select(pl.len()).collect().item()
print(f"Merged Bureau rows: {merged_rows}")

# Preview only — the full merged frame is ~25M rows, so we don't collect it all
bureau_merged_lf.head().collect()

In [ ]:
# Materialize a bounded preview instead of collecting all ~25M rows.
# Use .collect() explicitly only if you truly need the whole frame in memory.
bureau_merged_lf.head(100).collect()

In [ ]:
bureau_lf.filter(pl.col("SK_ID_CURR") == 215354).collect()

In [ ]:
new_df_lf = bureau_lf.filter(pl.col("SK_ID_CURR") == 215354).join(
    bureau_balance_lf, on="SK_ID_BUREAU", how="left"
)

In [ ]:
new_df_lf.collect()

In [ ]:
# Rows where the left join found no matching bureau_balance record.
# Filtering lazily lets polars push the predicate down before materializing.
bureau_merged_lf.filter(pl.col("STATUS").is_null()).collect()

## 5. Bureau Balance Join Coverage

How completely does `bureau_balance` cover each customer's bureau credits? A
`SK_ID_BUREAU` is *matched* if it has any rows in `bureau_balance`; unmatched
credits are exactly the ones that produce null `STATUS`/`MONTHS_BALANCE` in the
left join.

In [ ]:
# Flag each bureau credit with whether it has any bureau_balance history.
# Joining on the DISTINCT balance ids avoids exploding the ~25M-row row-level join.
balance_ids = (
    bureau_balance_lf.select("SK_ID_BUREAU")
    .unique()
    .with_columns(pl.lit(True).alias("has_balance"))
)

bureau_flagged_lf = bureau_lf.join(
    balance_ids, on="SK_ID_BUREAU", how="left"
).with_columns(pl.col("has_balance").fill_null(False))

In [ ]:
# Part 1: sample of (SK_ID_CURR, SK_ID_BUREAU) with no bureau_balance match
# (exactly the rows where STATUS and MONTHS_BALANCE come out null in the join)
bureau_flagged_lf.filter(~pl.col("has_balance")).select(
    "SK_ID_CURR", "SK_ID_BUREAU"
).head(20).collect()

In [ ]:
# Part 2: per-customer join coverage
per_curr = (
    bureau_flagged_lf.group_by("SK_ID_CURR")
    .agg(
        n_bureau=pl.len(),
        n_matched=pl.col("has_balance").sum(),
    )
    .collect()
)

total = per_curr.height
none_matched = per_curr.filter(pl.col("n_matched") == 0).height
all_matched = per_curr.filter(pl.col("n_matched") == pl.col("n_bureau")).height
partial = total - none_matched - all_matched

print(f"Total distinct SK_ID_CURR in bureau: {total}")
print(
    f"1) SK_ID_CURR with NO bureau_balance match on any SK_ID_BUREAU: "
    f"{none_matched} ({none_matched / total * 100:.2f}%)"
)
print(
    f"2) SK_ID_CURR with a bureau_balance match on ALL its SK_ID_BUREAU: "
    f"{all_matched} ({all_matched / total * 100:.2f}%)"
)
print(
    f"   partial (some but not all matched): {partial} ({partial / total * 100:.2f}%)"
)